In [2]:
# ==============================================================================
# WF6302 - LightGBM: Fine-Tuning de Aprendizaje Lento y Cobertura de Variables
# ==============================================================================

# 1. Limpieza y preparación del entorno
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

require("data.table")
if (!require("R.utils")) install.packages("R.utils")
require("R.utils")
if (!require("yaml")) install.packages("yaml")
require("yaml")

# 2. Configuración de parámetros iniciales
PARAM <- list()
PARAM$semilla_primigenia <- 130003
PARAM$experimento <- 6302
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# Creación de carpetas de experimento
setwd("/home/administrador/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings = FALSE)
setwd(paste0("/home/administrador/buckets/b1/exp/", experimento_folder))

# 3. Carga y preprocesamiento del Dataset
dataset <- fread(paste0("/home/administrador/buckets/b1/datasets/", PARAM$dataset))

# Catastrophe Analysis
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# Feature Engineering básico e histórico (Lags y Deltas)
divseg <- function(n, d) {
  d[d == 0] <- NA
  return(n / d)
}

if ("foto_mes" %in% colnames(dataset)) dataset[, kmes := foto_mes %% 100]

cols_lagueables <- copy(setdiff(
  colnames(dataset), 
  c("numero_de_cliente", "foto_mes", "clase_ternaria")
))

dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]

for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# 4. Estrategia de entrenamiento
PARAM$trainingstrategy$validate <- c(202107)
PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)
PARAM$trainingstrategy$training_pct <- 1.0
PARAM$trainingstrategy$positivos <- c("BAJA+1", "BAJA+2")

dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0)]
campos_buenos <- copy(setdiff(colnames(dataset), c("clase_ternaria", "clase01", "azar")))

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]
dataset[, fold_train := foto_mes %in% PARAM$trainingstrategy$training & 
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") | azar < PARAM$trainingstrategy$training_pct)]

if (!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label = dataset[fold_train == TRUE, clase01],
  free_raw_data = TRUE
)

dvalidate <- lgb.Dataset(
  data = data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label = dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data = TRUE
)

# 5. Hyperparameter Tuning (Grid Search con Aprendizaje Lento)
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "auc",
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 63, # Mayor resolución para variables continuas
  learning_rate = 0.01,
  num_iterations = 3000,
  early_stopping_rounds = 300
)

Estimar_AUC_lightgbm <- function(x) {
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  modelo_train <- lgb.train(
    data = dtrain,
    valids = list(valid = dvalidate),
    eval = "auc",
    param = param_completo,
    verbose = -100
  )
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full = TRUE, verbose = FALSE)
  return(list(AUC, niter))
}

# Búsqueda en grilla orientada a mayor cobertura estructural y de variables
tb_nueva <- CJ(
  num_leaves = c(31, 63, 127),
  max_depth = c(6, 8, 10),
  min_data_in_leaf = c(200, 800),
  feature_fraction = c(0.7, 0.9),
  pos_bagging_fraction = c(0.7, 1.0)
)

tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(.SD), by = 1:nrow(tb_nueva)]

fwrite(tb_nueva, file = "tb_grid_search_6302.txt", sep = "\t")
setorder(tb_nueva, -AUC)

PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_nueva[1])
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL

# 6. Modelo Final y Predicción
PARAM$trainingstrategy$final_train <- c(
  202107, 202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007, 202006, 202005
)
dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train]

dfinal_train <- lgb.Dataset(
  data = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
  label = dataset[fold_final_train == TRUE, clase01],
  free_raw_data = TRUE
)

fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

final_model <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
lgb.save(final_model, "modelo_6302.txt")

# Scoring
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
prediccion <- predict(final_model, data.matrix(dfuture[, campos_buenos, with = FALSE]))

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]
fwrite(tb_prediccion, file = "prediccion_6302.txt", sep = "\t")

# Envíos a Kaggle
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)
setorder(tb_prediccion, -prob)
dir.create("kaggle", showWarnings = FALSE)

for (envios in PARAM$kaggle$cortes) {
  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]
  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")
}

write_yaml(PARAM, file = "PARAM_6302.yml")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,743665,39.8,1479561,79.1,1479561,79.1
Vcells,1365728,10.5,8388608,64.0,2165301,16.6


Cargando paquete requerido: lightgbm



In [3]:
# 6. Generación de envíos y Submit a Kaggle especificando el binario
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)
setorder(tb_prediccion, -prob)
dir.create("kaggle", showWarnings = FALSE)

# Ruta absoluta al ejecutable de Kaggle en tu venv
kaggle_bin <- "/home/administrador/.venv/bin/kaggle"

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  # Armado de la línea de comandos apuntando a la ruta del venv
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'envios=", envios, " semilla=", PARAM$semilla_primigenia, "'")

  linea <- paste(kaggle_bin, "competitions submit", competencia, arch, mensaje)

  cat("Ejecutando:", linea, "\n")
  
  Sys.sleep(10) # Pausa para no saturar peticiones
  salida <- system(linea, intern = TRUE) # Ejecución con el binario correcto
  cat(salida, "\n\n")
}

write_yaml(PARAM, file = "PARAM_6301.yml")

Ejecutando: /home/administrador/.venv/bin/kaggle competitions submit -c utn-2026-virtual-mgr -f ./kaggle/KA6302_800.csv -m 'envios=800 semilla=130003' 
99 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 

Ejecutando: /home/administrador/.venv/bin/kaggle competitions submit -c utn-2026-virtual-mgr -f ./kaggle/KA6302_850.csv -m 'envios=850 semilla=130003' 
98 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 

Ejecutando: /home/administrador/.venv/bin/kaggle competitions submit -c utn-2026-virtual-mgr -f ./kaggle/KA6302_900.csv -m 'envios=900 semilla=130003' 
97 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 

Ejecutando: /home/administrador/.venv/bin/kaggle competitions submit -c utn-2026-virtual-mgr -f ./kaggle/KA6302_950.csv -m 'envios=950 semilla=130003' 
96 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 

Ejecutando: /home/administrador/.venv/bin/kaggle competitions submit

In [4]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [5]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "lun sep 21 00:25:18 2026"